# Project 1: Student Score Predictor
**Задача:** предсказать оценку студента по часам учёбы и посещаемости (regression).
**Темы:** linear regression, train/test split, MAE/R2, визуализация.

## План
1. Генерация данных → 2. EDA (исследование) → 3. X и y → 4. Split → 5. Обучение → 6. Метрики → 7. Графики → 8. Выводы

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Данные: 150 студентов
rng = np.random.default_rng(42)
n = 150
df = pd.DataFrame({
    "часы_учёбы":    rng.uniform(0, 10, n).round(1),
    "посещаемость":  rng.uniform(40, 100, n).round(0),   # % посещённых пар
})
df["оценка"] = (5*df["часы_учёбы"] + 0.4*df["посещаемость"]
                + rng.normal(0, 6, n)).clip(0, 100).round(0)
df.head()

In [ ]:
# 2. EDA — смотрим на данные глазами
print(df.shape)
print(df.describe().round(1))
print("Пропуски:", df.isna().sum().sum())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(df["часы_учёбы"], df["оценка"], s=12); axes[0].set_xlabel("часы"); axes[0].set_ylabel("оценка")
axes[1].scatter(df["посещаемость"], df["оценка"], s=12, c='g'); axes[1].set_xlabel("посещаемость")
plt.show()
# Видно: оба признака связаны с оценкой положительно — модели есть что ловить

In [ ]:
# 3-5. X, y -> split -> обучение
X = df[["часы_учёбы", "посещаемость"]]
y = df["оценка"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression().fit(X_tr, y_tr)
y_pred = model.predict(X_te)

# 6. Метрики
print(f"MAE = {mean_absolute_error(y_te, y_pred):.1f} баллов")
print(f"R2  = {r2_score(y_te, y_pred):.3f}")
for name, w in zip(X.columns, model.coef_):
    print(f"вес {name}: {w:+.2f}")

In [ ]:
# 7. Визуализация качества + error analysis
plt.scatter(y_te, y_pred, alpha=0.7)
lims = [0, 100]; plt.plot(lims, lims, 'r--')
plt.xlabel('реальная оценка'); plt.ylabel('предсказанная'); plt.grid(True); plt.show()

residuals = y_te - y_pred
plt.hist(residuals, bins=15); plt.title('Остатки'); plt.grid(True); plt.show()
worst = residuals.abs().sort_values(ascending=False).head(3)
print("Самые большие промахи (индексы и величина):"); print(worst)

## Выводы
- Модель промахивается в среднем на ~5 баллов (MAE), R2 ~0.85+ — закономерность поймана.
- Вес часов учёбы ~5: каждый час даёт ~5 баллов. Посещаемость тоже помогает, но слабее.
- Остатки симметричны вокруг 0 — систематической ошибки нет.

## Задания для практики
1. Добавь признак «часы сна» (влияет слабо) — изменился ли R2?
2. Сделай предсказание для себя: свои часы и посещаемость.
3. Замени LinearRegression на DecisionTreeRegressor(max_depth=4) — лучше или хуже?